## Библиотеки и константы


In [71]:
import pandas as pd
import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
import random
import scipy.stats as stats
import pylab
import struct
import requests
import json
import os
from IPython.display import clear_output
from math import factorial
from collections import defaultdict
from itertools import product, combinations, permutations
import inspect
import time

%matplotlib inline

### Базовые значения

In [72]:
stats_map = {
    "HP%": 0,
    "HP": 1, 
    "ATK%": 2,
    "ATK": 3,
    "DEF%": 4,
    "DEF": 5,
    "PEN": 6,
    "AP": 7, # Anomaly Proficiency
    "CRIT_RATE": 8,
    "CRIT_DMG": 9,
    "PEN%": 10,
    "AM": 11,
    "Impact": 12,
    "ER": 13,
    "Physical Bonus": 14,
    "Fire Bonus": 15,
    "Ice Bonus": 16,
    "Electric Bonus": 17,
    "Ether Bonus": 18,
    "Wind Bonus": 19,
    "None": 100,
}

stat_names = [""] * (max(stats_map.values()) + 1)
for name, stat_id in stats_map.items():
    if stat_id >= 0:
        stat_names[stat_id] = name

sub_stats_weights = np.array([
    113, # HP%
    103, # HP
    90, # ATK%
    98, # ATK
    131, # DEF%
    112, # DEF
    107, # PEN
    97, # AP
    116, # CRIT_RATE
    104, # CRIT_DMG
              
], dtype=np.int32)


file_names = {'HP%', 'HP', 'ATK%', 'ATK', 'DEF%', 'DEF', 'AP', "CRIT_RATE", "CRIT_DMG", "None"}
possible_main_stats = [stats_map[name] for name in file_names]

### Функции

In [73]:
def get_roll_distributions(roll_count):
    result = []

    for r1 in range(roll_count + 1):
        for r2 in range(roll_count - r1 + 1):
            for r3 in range(roll_count - r1 - r2 + 1):
                r4 = roll_count - r1 - r2 - r3

                rolls = (r1, r2, r3, r4)

                combinations = (
                    factorial(roll_count)
                    // (
                        factorial(r1)
                        * factorial(r2)
                        * factorial(r3)
                        * factorial(r4)
                    )
                )

                probability = combinations / (4 ** roll_count)

                result.append((rolls, probability))

    return result

roll_distributions_4 = get_roll_distributions(4)
roll_distributions_5 = get_roll_distributions(5)

In [74]:
def get_substat_set_probability(substats, available_stats):
    total_probability = 0

    for order in permutations(substats):
        remaining = list(available_stats)
        probability = 1

        for stat in order:
            total_weight = sum(
                sub_stats_weights[s]
                for s in remaining
            )

            probability *= (
                sub_stats_weights[stat]
                / total_weight
            )

            remaining.remove(stat)

        total_probability += probability

    return total_probability

def get_substat_distributions(main_stat):
    available_stats = list(range(10))

    if main_stat < 10:
        available_stats.remove(main_stat)

    distributions = []

    for substats in combinations(available_stats, 4):
        probability = get_substat_set_probability(
            substats,
            available_stats
        )

        distributions.append((substats, probability))

    return distributions

def generate_all_discs(main_stat):
    substat_distributions = get_substat_distributions(main_stat)

    roll_distributions = [
        (4, 0.8, roll_distributions_4),
        (5, 0.2, roll_distributions_5)
    ]

    discs = []
    probabilities = []

    for substats, substats_probability in substat_distributions:
        for _, roll_count_probability, rolls_list in roll_distributions:
            for rolls, rolls_probability in rolls_list:

                disc = [0] * 10

                for stat, roll in zip(substats, rolls):
                    disc[stat] = roll + 1

                probability = (
                    substats_probability
                    * roll_count_probability
                    * rolls_probability
                )

                discs.append(disc)
                probabilities.append(probability)

    return (
        np.array(discs, dtype=np.uint8),
        np.array(probabilities)
    )

def save_all_discs(main_stat):
    start_time = time.time()
    substats, probabilities = generate_all_discs(main_stat)
    
    assert np.isclose(probabilities.sum(), 1.0)

    os.makedirs("discs", exist_ok=True)

    stat_name = stat_names[main_stat]

    np.savez_compressed(
        f"discs/{stat_name}.npz",
        substats=substats,
        probabilities=probabilities
    )

    end_time = time.time()
    print(
        f"{stat_name}: "
        f"{len(substats)} discs, "
        f"probability={probabilities.sum()}"
        f"\n--- {end_time - start_time} seconds ---"
    )

In [75]:
def generate_disc_files():
    start_time = time.time()
    for main_stat in possible_main_stats:
        save_all_discs(main_stat)
    end_time = time.time()
    print("--- %s seconds of all disks ---" % (end_time - start_time))

In [76]:
generate_disc_files()

DEF: 11466 discs, probability=1.0
--- 0.039008378982543945 seconds ---
CRIT_DMG: 11466 discs, probability=1.0
--- 0.03600811958312988 seconds ---
HP: 11466 discs, probability=1.0
--- 0.038008928298950195 seconds ---
ATK%: 11466 discs, probability=1.0
--- 0.03700828552246094 seconds ---
ATK: 11466 discs, probability=1.0
--- 0.03900885581970215 seconds ---
AP: 11466 discs, probability=1.0
--- 0.06601500511169434 seconds ---
CRIT_RATE: 11466 discs, probability=1.0
--- 0.03600788116455078 seconds ---
HP%: 11466 discs, probability=1.0
--- 0.03700828552246094 seconds ---
None: 19110 discs, probability=1.0
--- 0.06301426887512207 seconds ---
DEF%: 11466 discs, probability=1.0
--- 0.03600788116455078 seconds ---
--- 0.42809557914733887 seconds of all disks ---
